# Fase 1b · Validar L-SHADE y jSO — Mac mini

Plan de implementación, Fase 1b (y 1c detrás de un flag).

1. Comprueba entorno y hashes de los archivos copiados.
2. Chequeo de presupuesto: en las 6 celdas del protocolo tienen que gastar exactamente (T+1)·N = 5040 / 50 080 / 500 080.
3. Validación contra CEC2017 D50 a 500 000 FES exactos (10 000·D), 25 corridas × 29 funciones × 2 algoritmos.
4. Reporte: media por función contra la publicada.
5. (Fase 1c, apagada) Protocolo de 51 corridas, 500k primero.

Copiar al mini: `competidores.py`, `validar_shade.py` y este notebook. Si se interrumpe, volver a ejecutar: `validar_shade.py` retoma donde quedó.

In [ ]:
import sys, os, hashlib, subprocess, opfunu

WORKERS = 20
ESPERADO = {
    "competidores.py":  "783a7ada89ddf917332d91efcf8179a8c3c703797943eb415d14a7141c68487b",
    "validar_shade.py": "a2275c7fe5c95c76439ee3c6d29d7e8537343e44318a5a5668e92980e9a92692",
}

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:20s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del MacBook: copiala de nuevo"
print("\nEntorno y archivos OK")

## 1 · Chequeo de presupuesto en las 6 celdas (~30 s)
Tiene que decir `ratio=1.0000` en las 12 líneas.

In [ ]:
import competidores as comp
from experimento_config import get_problems_for_dim, POP_SIZE

ok = True
for D in (10, 50):
    pr = get_problems_for_dim(D)[33]          # CEC2017 F5 oficial (Rastrigin rotado)
    for mf in (5000, 50000, 500000):
        T = mf // POP_SIZE; esperado = (T + 1) * POP_SIZE
        for alg in ("lshade", "jso"):
            r = getattr(comp, alg)(pr["f"], pr["bounds"], N=POP_SIZE, T=T, seed=0)
            ratio = r["fes"] / esperado
            ok &= ratio == 1.0
            print(f"D{D} {mf:>7,} {alg:6s} fes={r['fes']:>7,} ratio={ratio:.4f} gens={r['iters']:>6} err={r['best_f']:.3e}")
assert ok, "presupuesto distinto de (T+1)*N: no seguir, mandar la salida a Claude"
PRESUPUESTO_OK = True

## 2 · Validación CEC2017 D50, 500 000 FES (~15–30 min con 20 workers)
Sale en `resultados_experimento/validacion/`.

In [ ]:
assert globals().get("PRESUPUESTO_OK"), "primero el chequeo de presupuesto"
rc = correr("validar_shade.py", "run", "--runs", "25", "--workers", str(WORKERS))
assert rc == 0

## 3 · Reporte
Criterio: mismo orden de magnitud por función y mismo patrón de resueltas (< 1e-8). Las medias publicadas del script están transcritas de memoria: confirmar contra los PDF. Traer al MacBook los 2 JSON `VALID_*` y la salida de esta celda.

In [ ]:
rc = correr("validar_shade.py", "report")

## 4 · Fase 1c — protocolo de 51 corridas (apagado)
Sólo si la validación pasó. Orden: 500k (D10, D50), luego 50k, luego 5k. Con los dos algoritmos, las celdas de 500k llevan más de un día.
Salida: `resultados_experimento/{LSHADE,JSO}_D{10,50}_FES{5000,50000,500000}.json` (3621 entradas cada uno).

In [ ]:
RUN_PROTOCOLO = False

if RUN_PROTOCOLO:
    from experimento_config import run_experiment
    for mf in (500_000, 50_000, 5_000):
        for D in (10, 50):
            for alg in ("LSHADE", "JSO"):
                run_experiment(alg, None, D, mf, n_workers=WORKERS)

In [ ]:
import json, math
for alg in ("LSHADE", "JSO"):
    for D in (10, 50):
        for mf in (5000, 50000, 500000):
            p = f"resultados_experimento/{alg}_D{D}_FES{mf}.json"
            if not os.path.exists(p):
                print(f"FALTA   {p}"); continue
            v = json.load(open(p))
            malos = sum(1 for x in v.values() if not math.isfinite(x))
            print(f"{'OK     ' if len(v) == 71*51 and not malos else 'REVISAR'} {os.path.basename(p)}: "
                  f"{len(v)}/{71*51} corridas, {malos} no finitos")